In [ ]:
import pandas as pd
DATA_DIR = r"C:\Users\HARDPC\Desktop\AL\projekty\products_prices_historical_analysis\keepa_data"

In [ ]:
meta = pd.read_csv(DATA_DIR + r"\all_products_meta.csv")

meta.shape
meta.info()

KEEPA_EPOCH = pd.Timestamp("2011-01-01")
meta['listed_since'] = KEEPA_EPOCH + pd.to_timedelta(meta['listedSince'], unit = 'min')
meta['tracking_since'] = KEEPA_EPOCH + pd.to_timedelta(meta['trackingSince'], unit = 'min')

meta["tracking_gap_days"] = (meta["tracking_since"] - meta["listed_since"]).dt.days
meta[["asin", "brand", "model", "listed_since", "tracking_since", "tracking_gap_days"]].sort_values("tracking_gap_days", ascending=False)


In [ ]:
meta.info()
KEEPA_EPOCH = pd.Timestamp("2011-01-01")
meta['listed_since'] = KEEPA_EPOCH + pd.to_timedelta(meta['listedSince'], unit = 'min')
meta['tracking_since'] = KEEPA_EPOCH + pd.to_timedelta(meta['trackingSince'], unit = 'min')
meta["tracking_gap_days"] = (meta["tracking_since"] - meta["listed_since"]).dt.days
meta[["asin", "brand", "model", "listed_since", "tracking_since", "tracking_gap_days"]].sort_values("tracking_gap_days", ascending=False)


In [ ]:
ph = pd.read_csv(DATA_DIR + r"\B09LNW3CY2_price_history.csv")
ph.shape
ph.head(10)
ph.info


price_cols = ["AMAZON", "NEW", "USED", "LISTPRICE", "REFURBISHED"]
ph[price_cols].notna().sum()


for asin in ["B09VH9BKHS", "B0BL8HPF13"]:
    ph_tmp = pd.read_csv(DATA_DIR + f"\\{asin}_price_history.csv")
    print(asin)
    print(ph_tmp[price_cols].notna().sum())
    print()

In [ ]:
#price history data loading - combining all price history .csv files into one DF - os + pandas
#note - data/ files are not shared in repo, as it would obviously be very ineffective, samples in data_schema_examples.md
import os
import pandas as pd

df = pd.DataFrame()
df2 = pd.DataFrame()
data_path = os.path.join('../data/')
for file in os.listdir(data_path):
    if '_price_history' in file:
        df2 = pd.read_csv(data_path + file)
        df = pd.concat([df, df2])


In [81]:
#data resampling - from the loaded 
filtered_df = df[df['NEW'].notna()]
filtered_df['datetime'] = pd.to_datetime(filtered_df['datetime'])
filtered_df = filtered_df.set_index('datetime', drop = True)

pre_filtered_df = filtered_df.groupby("asin").resample("W")["NEW"].min()
print(pre_filtered_df)
filtered_df = filtered_df.groupby("asin").resample("W")["NEW"].min().reset_index()
print(filtered_df)
print(filtered_df['NEW'].notna().sum()) #There are still 78 NaN values (out of 3062 values, 2.5% - it's alright), which means in case of 78 weeks in total we have no price points.
#I thought about using a mean of previous/next row, but we can also assume that the price DID NOT CHANGE during that time, so using ffill makes more sense here.

filtered_df['NEW'] = filtered_df.groupby('asin')['NEW'].ffill()
print(filtered_df['NEW'].notna().sum())
print(filtered_df.columns)
print(filtered_df.head())

asin        datetime  
B07ZPKN6YR  2019-11-10    649.99
            2019-11-17    649.99
            2019-11-24    649.99
            2019-12-01    659.99
            2019-12-08    659.99
                           ...  
B0FRKS39NK  2026-02-15    827.99
            2026-02-22    761.02
            2026-03-01    730.14
            2026-03-08    782.99
            2026-03-15    771.82
Name: NEW, Length: 3062, dtype: float64
            asin   datetime     NEW
0     B07ZPKN6YR 2019-11-10  649.99
1     B07ZPKN6YR 2019-11-17  649.99
2     B07ZPKN6YR 2019-11-24  649.99
3     B07ZPKN6YR 2019-12-01  659.99
4     B07ZPKN6YR 2019-12-08  659.99
...          ...        ...     ...
3057  B0FRKS39NK 2026-02-15  827.99
3058  B0FRKS39NK 2026-02-22  761.02
3059  B0FRKS39NK 2026-03-01  730.14
3060  B0FRKS39NK 2026-03-08  782.99
3061  B0FRKS39NK 2026-03-15  771.82

[3062 rows x 3 columns]
2984
3062
Index(['asin', 'datetime', 'NEW'], dtype='str')
         asin   datetime     NEW
0  B07ZPKN6YR 2019-11-10  